# 03 — Téléchargement des PDF PTR House

**Rôle :** récupérer tous les PDF PTR listés dans `house_ptr_index.csv`, de façon **relançable**, produire un manifest de complétude et un audit de qualité texte. Les PDF illisibles vont au **backlog OCR**.

**Prérequis :** `02_house_index` exécuté · `pip install pdfplumber`.

**Cellule 1 — Imports & chemins.**

In [ ]:
import requests, time
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT  = find_project_root(Path.cwd())
PROC_HOUSE    = PROJECT_ROOT / "data" / "processed" / "house"
RAW_HOUSE_PDF = PROJECT_ROOT / "data" / "raw" / "house" / "ptr_pdfs"; RAW_HOUSE_PDF.mkdir(parents=True, exist_ok=True)
AUDIT         = PROJECT_ROOT / "data" / "audit"; AUDIT.mkdir(parents=True, exist_ok=True)
REPORTS       = PROJECT_ROOT / "reports"; REPORTS.mkdir(parents=True, exist_ok=True)
REQUEST_PAUSE_S = 0.6
USER_AGENT = "Ramify-QIS research (contact: <ton-email>)" 

**Cellule 2 — Chargement de la checklist PTR** produite au notebook 02.

In [ ]:
ptr = pd.read_csv(PROC_HOUSE / "house_ptr_index.csv", dtype={"doc_id": str})
print("PTR à récupérer :", len(ptr))
ptr.head(3)

**Cellule 3 — Téléchargement d'un PDF (idempotent).** Rangé par année ; saute si déjà présent.

In [ ]:
def download_pdf(year, doc_id, url):
    folder = RAW_HOUSE_PDF / str(int(year)); folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{doc_id}.pdf"
    if path.exists() and path.stat().st_size > 0:
        return "exists", path
    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=60)
        if r.status_code != 200 or not r.content:
            return f"http_{r.status_code}", path
        path.write_bytes(r.content)
        return "ok", path
    except Exception:
        return "error", path

**Cellule 4 — Boucle de téléchargement.** Réglez `MAX_FILES` (ex. 20) pour un test, `None` pour tout (~8 000 PDF). Relançable.

In [ ]:
MAX_FILES = 20   # -> None pour le run complet

rows = ptr if MAX_FILES is None else ptr.head(MAX_FILES)
status = []
for _, r in rows.iterrows():
    st, path = download_pdf(r["year"], r["doc_id"], r["pdf_url"])
    status.append({"year": r["year"], "doc_id": r["doc_id"], "status": st, "path": str(path)})
    if st == "ok":
        time.sleep(REQUEST_PAUSE_S)
manifest = pd.DataFrame(status)
print(manifest["status"].value_counts())

**Cellule 5 — Manifest de complétude.** Attendu vs obtenu.

In [ ]:
manifest["present"] = manifest["status"].isin(["ok", "exists"])
manifest.to_csv(AUDIT / "house_pdf_manifest.csv", index=False)
print("Présents :", int(manifest["present"].sum()), "/", len(manifest))

**Cellule 6 — Audit de qualité texte.** Les PDF sans texte exploitable → backlog OCR.

In [ ]:
import pdfplumber

def has_text(path):
    try:
        with pdfplumber.open(path) as pdf:
            txt = "".join((p.extract_text() or "") for p in pdf.pages[:2])
        return len(txt.strip()) > 50
    except Exception:
        return False

present = manifest[manifest["present"]].copy()
present["ok_text"] = present["path"].map(has_text)
present.to_csv(AUDIT / "house_pdf_text_quality.csv", index=False)
n_ok = int(present["ok_text"].sum()); n_tot = len(present)
print(f"Texte exploitable : {n_ok}/{n_tot}  |  backlog OCR : {n_tot - n_ok}")

**Cellule 7 — Rapport téléchargement + qualité.**

In [ ]:
from datetime import datetime, timezone
lines = ["# Audit téléchargement PDF House", "",
         f"Généré : {datetime.now(timezone.utc).isoformat(timespec='seconds')}", "",
         f"- PDF du lot : {len(manifest)}",
         f"- Présents : {int(manifest['present'].sum())}",
         f"- Texte exploitable : {n_ok}",
         f"- Backlog OCR (no_text) : {n_tot - n_ok}"]
(REPORTS / "house_download_audit.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

PDF récupérés ✅ — passez à **`04_house_parse`**.